<div align="center">

# ⚛️ GOAL — Getting Started Tutorial

### *From zero to a trained interatomic potential in one notebook*

</div>

---

**Welcome!** This notebook is a step-by-step guide for beginners who want to use
[**GOAL**](https://github.com/Nourollah/GOAL) (General Open Atomistic Laboratory)
as a Python package — no YAML config files, no command-line tools. Everything
happens right here in code you can read and run.

**What you will learn:**

| # | Topic | What you will do |
|---|-------|------------------|
| 1 | Installation & imports | Install GOAL and import the key modules |
| 2 | Project tour | Understand the moving parts: backbones, heads, losses, data |
| 3 | Monolithic model | Build a self-contained model that predicts energy & forces |
| 4 | Modular model | Compose a backbone + head, the recommended GOAL workflow |
| 5 | Dataset loading | Read `.traj` files and convert them to GOAL's graph format |
| 6 | DataLoaders | Batch, shuffle, and split the data for training |
| 7 | Training setup | Wire up optimiser, loss, and the training module |
| 8 | Training loop | Train for a few epochs and watch the loss go down |
| 9 | Checkpointing | Save and reload a trained model |
| 10 | ASE Calculator | Use your model as an ASE calculator for energy & forces |
| 11 | MD / Optimisation | Run a geometry optimisation or short MD with the calculator |

> **Prerequisites:** Basic Python and some familiarity with PyTorch. No prior
> knowledge of atomistic simulation or graph neural networks is assumed — we
> explain everything as we go.

> **Hardware:** A CPU is sufficient for this tutorial. If you have a GPU,
> training will be faster, but nothing here requires one.

---

## 1 · Install and Import the Package

If you have already installed GOAL, skip the install cell. Otherwise, run it
once — it installs GOAL in *editable* mode so any code changes take effect
immediately.

> **Tip:** If you are using pixi, run `pixi install` in a terminal instead.

In [ ]:
# Uncomment the line below if GOAL is not yet installed:
# !pip install -e "/home/masoud/Projects/GOAL"

Now let's import everything we will use throughout this notebook. Don't worry
about memorising these — we will explain each one when we first use it.

| Import | What it does |
|--------|-------------|
| `AtomicGraph` | The universal data object — every model speaks this language |
| `NodeFeatures` | What a backbone produces: per-atom feature vectors |
| `DeepSet` | An invariant backbone (feature extractor) |
| `EnergyForcesHead` | A head that turns features into energy & forces |
| `MonolithicExample` | A minimal self-contained model (no separate head) |
| `GOALModule` | Lightning module that wires backbone + head + loss |
| `CompositeLoss`, `WeightedLoss` | Composable loss system |
| `GOALCalculator` | ASE calculator wrapper for any trained model |

In [ ]:
from __future__ import annotations

# ── Standard library & PyTorch ──────────────────────────────────────
import typing
from pathlib import Path

import torch
import torch.nn as nn

# ── GOAL: data ──────────────────────────────────────────────────────
from goal.ml.data.graph import AtomicGraph, NodeFeatures

# ── GOAL: model components ──────────────────────────────────────────
from goal.ml.nn.models.deepset import DeepSet                    # invariant backbone
from goal.ml.nn.heads.energy_forces import EnergyForcesHead      # task head
from goal.ml.nn.models.monolithic_example import MonolithicExample  # monolithic demo

# ── GOAL: training ──────────────────────────────────────────────────
from goal.ml.training.module import GOALModule
from goal.ml.training.loss import (
    CompositeLoss,
    WeightedLoss,
    EnergyLoss,
    ForcesLoss,
)

# ── GOAL: ASE integration ──────────────────────────────────────────
from goal.ml.utils.calculator import GOALCalculator

# ── ASE (Atomic Simulation Environment) ─────────────────────────────
import ase
from ase import Atoms
from ase.io import read as ase_read, write as ase_write
from ase.build import molecule, bulk
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

# ── PyG (graph batching) ───────────────────────────────────────────
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader as PyGDataLoader

print(f"✓ PyTorch  {torch.__version__}")
print(f"✓ ASE      {ase.__version__}")
print(f"✓ GOAL     imported successfully")

---

## 2 · Understanding the Project Structure

Before writing any model code, let's get a bird's-eye view of how GOAL is
organised. Think of GOAL as a LEGO set with four kinds of pieces:

```
┌─────────────────────────────────────────────────────────┐
│                    GOAL Architecture                     │
│                                                         │
│   ┌──────────┐    ┌──────────────┐    ┌──────────┐     │
│   │ Dataset  │───►│  AtomicGraph │───►│ Backbone │     │
│   │ (.traj)  │    │ (positions,  │    │ (DeepSet,│     │
│   │          │    │  z, edges)   │    │  GNN...) │     │
│   └──────────┘    └──────────────┘    └────┬─────┘     │
│                                            │            │
│                                     NodeFeatures        │
│                                     (per-atom           │
│                                      vectors)           │
│                                            │            │
│                                       ┌────▼─────┐     │
│                                       │   Head   │     │
│                                       │ (energy, │     │
│                                       │  forces) │     │
│                                       └────┬─────┘     │
│                                            │            │
│                                    dict[str, Tensor]    │
│                                    {"energy": ...,      │
│                                     "forces": ...}      │
│                                            │            │
│                                       ┌────▼─────┐     │
│                                       │   Loss   │     │
│                                       │ compare  │     │
│                                       │ to truth │     │
│                                       └──────────┘     │
└─────────────────────────────────────────────────────────┘
```

**Key concepts:**

- **`AtomicGraph`** — A PyTorch Geometric `Data` object holding positions,
  atomic numbers, edge indices, edge vectors, and (optionally) training
  targets like energy and forces. This is the *only* data format inside the
  training loop — no ASE objects, no numpy arrays.

- **Backbone** — A neural network that takes an `AtomicGraph` and produces
  `NodeFeatures` (per-atom learned representations). Examples: `DeepSet`,
  `InvariantGNN`, `HyperSpec`.

- **Head** — A small network that takes `NodeFeatures` + the graph and
  predicts physical properties. Example: `EnergyForcesHead` predicts energy
  as a sum of per-atom contributions and forces via autograd.

- **Monolithic model** — An alternative that merges backbone + head into one
  class. Simpler, but less flexible (you can't swap heads).

- **`GOALModule`** — A Lightning module that wires backbone + head + loss
  together and handles the training loop automatically.

Let's explore what's available in the model registries:

In [ ]:
from goal.ml.registry import MODEL_REGISTRY, HEAD_REGISTRY, LOSS_REGISTRY

print("═══ Available Backbones / Models ═══")
for name in sorted(MODEL_REGISTRY.keys()):
    print(f"  • {name}")

print("\n═══ Available Heads ═══")
for name in sorted(HEAD_REGISTRY.keys()):
    print(f"  • {name}")

print("\n═══ Available Loss Components ═══")
for name in sorted(LOSS_REGISTRY.keys()):
    print(f"  • {name}")

---

## 3 · Building a Model from Scratch — Monolithic Approach

A **monolithic** model is the simplest way to get started. It handles
everything internally: embedding atoms, computing interactions, and
producing energy and forces — all in a single `forward()` call.

Think of it like ordering a complete meal at a restaurant: you don't pick
individual ingredients, you just say "I want the set menu" and get
everything together.

**When to use monolithic:**
- You have your own custom architecture and just want to use GOAL's
  training infrastructure
- You want a simple, self-contained model without worrying about
  separating concerns

Let's create one using the built-in `MonolithicExample`:

In [ ]:
# ── Create a monolithic model ────────────────────────────────────────
# All hyperparameters are passed directly as keyword arguments.
# No config files needed!

monolithic_model: nn.Module = MonolithicExample(
    num_elements=120,    # support up to element 120 (covers the whole periodic table)
    embedding_dim=64,    # each atom type gets a 64-dimensional learned embedding
    hidden_dim=64,       # width of the readout MLP hidden layer
)

# Let's see what's inside
print(monolithic_model)
print(f"\nTotal parameters: {sum(p.numel() for p in monolithic_model.parameters()):,}")

**Reading the output above:**

- `embedding` — A lookup table that maps each atomic number (1=H, 6=C, 8=O, etc.)
  to a learned vector. The model learns what each element "means" during training.
- `readout` — A small MLP (Multi-Layer Perceptron) that takes the embedding and
  predicts a single number: the atom's energy contribution.
- Forces are computed automatically via PyTorch's autograd: $\mathbf{F} = -\nabla_{\mathbf{R}} E$

The `output_keys` property tells us what this model produces:

In [ ]:
print(f"This model produces: {monolithic_model.output_keys}")
# → ['energy', 'forces']

---

## 4 · Building a Model from Scratch — Modular Approach

The **modular** approach is GOAL's recommended workflow. Instead of one
monolithic class, you compose two separate pieces:

1. **Backbone** — extracts per-atom features from the atomic graph
2. **Head** — turns those features into physical predictions (energy, forces, etc.)

This is like cooking with separate courses: you can swap the main course
(backbone) while keeping the same dessert (head), or vice versa. Want
to switch from a simple invariant model to a fancy equivariant one? Just
change the backbone — the head stays the same.

```
AtomicGraph ──► Backbone ──► NodeFeatures ──► Head ──► {"energy": ..., "forces": ...}
                (DeepSet)    (128-dim vectors) (EnergyForcesHead)
```

Let's build a modular model using `DeepSet` (backbone) + `EnergyForcesHead` (head):

In [ ]:
# ── Step 1: Create the backbone ─────────────────────────────────────
# DeepSet is an invariant backbone — it only uses scalar features
# (distances, not directions), which makes it fast and simple.

backbone: nn.Module = DeepSet(
    num_elements=120,       # periodic table coverage
    embedding_dim=64,       # atom embedding dimension
    hidden_channels=64,     # width of internal layers (= output feature dim)
    num_filters=64,         # projected feature space for atoms & distances
    num_radial_basis=20,    # Bessel radial basis functions for distance encoding
    transform_depth=2,      # depth of projection MLPs
    cutoff=5.0,             # atoms beyond 5 Å are considered non-interacting
)

print("Backbone:")
print(backbone)
print(f"\nBackbone output irreps: {backbone.irreps_out}")
print(f"  → This means {backbone.hidden_dim} scalar (invariant) channels")
print(f"\nBackbone parameters: {sum(p.numel() for p in backbone.parameters()):,}")

In [ ]:
# ── Step 2: Create the head ─────────────────────────────────────────
# The head takes NodeFeatures and predicts energy & forces.
# We must tell it the shape of incoming features — that's irreps_in.

head: nn.Module = EnergyForcesHead(
    irreps_in=str(backbone.irreps_out),  # must match backbone output!
    hidden_dim=32,                        # readout MLP hidden width
    compute_stress=False,                 # we don't need stress for molecules
)

print("Head:")
print(head)
print(f"\nHead output keys: {head.output_keys}")
print(f"Head parameters:  {sum(p.numel() for p in head.parameters()):,}")

**Comparing the two approaches:**

| | Monolithic | Modular (backbone + head) |
|---|:---:|:---:|
| Number of classes | 1 | 2 |
| Can swap heads? | ❌ | ✅ |
| Can swap backbones? | ❌ | ✅ |
| Ideal for | Quick experiments, custom architectures | Production, research, mixing components |

For the rest of this tutorial, **we'll use the modular approach** — it's the
recommended workflow and you'll see why it's so flexible.

---

## 5 · Loading a Custom Dataset from `.traj` Files

Real-world data usually comes from DFT (Density Functional Theory)
calculations stored as `.traj` (ASE trajectory) files. Each frame in a
trajectory contains:
- **Positions** — where each atom is (x, y, z in Ångström)
- **Atomic numbers** — what each atom is (1=H, 6=C, 8=O, etc.)
- **Energy** — the total potential energy (in eV)
- **Forces** — the force on each atom (in eV/Å)
- **Cell** — the unit cell (for periodic systems like crystals)

Since you might not have a `.traj` file handy, we'll **create a small
synthetic dataset** first, then show how to load it. In your real work,
you'd skip the creation step and just point to your own files.

### 5a · Create a synthetic `.traj` dataset

We'll build a few water molecules with random perturbations and
assign fake energies/forces. This is just for demonstration — your
real data would come from actual DFT calculations.

In [ ]:
import numpy as np
from ase.io.trajectory import Trajectory

# ── Create a synthetic trajectory file ──────────────────────────────
# We'll make 100 slightly-perturbed water molecules as our "dataset".
# In real life, these would come from your DFT calculations.

traj_path: Path = Path("demo_dataset.traj")
rng: np.random.Generator = np.random.default_rng(seed=42)

with Trajectory(str(traj_path), mode="w") as traj:
    for i in range(100):
        # Start from equilibrium water geometry and add small random noise
        atoms: Atoms = molecule("H2O")
        atoms.positions += rng.normal(scale=0.05, size=atoms.positions.shape)

        # Assign synthetic "DFT" results as properties
        # (In reality, these come from your quantum chemistry calculation)
        fake_energy: float = -76.0 + rng.normal(scale=0.1)  # eV
        fake_forces: np.ndarray = rng.normal(scale=0.5, size=(len(atoms), 3))  # eV/Å

        atoms.info["energy"] = fake_energy
        atoms.arrays["forces"] = fake_forces

        traj.write(atoms)

print(f"✓ Created {traj_path} with 100 structures")

### 5b · Load and inspect the trajectory

Now let's load the data back — this is exactly what you'd do with your
own `.traj` file from a DFT calculation.

In [ ]:
# ── Load all frames from the .traj file ─────────────────────────────
all_frames: list[Atoms] = ase_read(str(traj_path), index=":")
print(f"Loaded {len(all_frames)} structures from {traj_path}\n")

# ── Inspect the first frame ─────────────────────────────────────────
sample: Atoms = all_frames[0]
print(f"Chemical formula:  {sample.get_chemical_formula()}")
print(f"Number of atoms:   {len(sample)}")
print(f"Atomic numbers:    {sample.numbers}")
print(f"Positions (Å):\n{sample.positions}")
print(f"\nEnergy (eV):       {sample.info['energy']:.4f}")
print(f"Forces (eV/Å):\n{sample.arrays['forces']}")

### 5c · Convert ASE Atoms to GOAL's AtomicGraph format

GOAL doesn't work with ASE `Atoms` objects directly inside the training
loop — that would be too slow. Instead, we convert each structure into
an `AtomicGraph`, which is a pure-tensor graph representation optimised
for GPU computation.

The key step is `AtomicGraph.from_ase()` — it builds a neighbour list
(which atoms are close enough to interact?) and computes edge vectors
and distances. The `cutoff` parameter controls how far apart two atoms
can be and still be considered neighbours.

In [ ]:
# ── Convert all ASE Atoms → AtomicGraph ─────────────────────────────
cutoff: float = 5.0  # Ångström — same value we used for the backbone!

graphs: list[AtomicGraph] = []
for atoms in all_frames:
    graph: AtomicGraph = AtomicGraph.from_ase(
        atoms,
        cutoff=cutoff,
        energy=atoms.info["energy"],
        forces=atoms.arrays["forces"],
    )
    graphs.append(graph)

print(f"Converted {len(graphs)} structures to AtomicGraph\n")

# ── Inspect one graph ───────────────────────────────────────────────
g: AtomicGraph = graphs[0]
print(f"Atomic numbers (z):  {g.z}")
print(f"Positions shape:     {g.pos.shape}  → (num_atoms, 3)")
print(f"Edge index shape:    {g.edge_index.shape}  → (2, num_edges)")
print(f"Edge lengths shape:  {g.edge_weight.shape}  → (num_edges,)")
print(f"Energy:              {g.energy.item():.4f} eV")
print(f"Forces shape:        {g.forces.shape}  → (num_atoms, 3)")
print(f"\nNumber of edges:     {g.edge_index.shape[1]}")
print(f"  (each atom has ~{g.edge_index.shape[1] / len(g.z):.0f} neighbours within {cutoff} Å)")

---

## 6 · Preparing Data with DataLoaders

Neural networks don't train on one structure at a time — they process
**batches** (groups of structures) for efficiency. We need to:

1. **Split** the data into training and validation sets
2. **Create DataLoaders** that serve batches to the training loop

GOAL uses PyTorch Geometric's `DataLoader`, which knows how to batch
graph data correctly (it concatenates node and edge tensors and adds a
`batch` index to track which nodes belong to which graph).

In [ ]:
# ── Split into train / validation ────────────────────────────────────
# 80% for training, 20% for validation
n_train: int = int(0.8 * len(graphs))
n_val: int = len(graphs) - n_train

# Shuffle before splitting (reproducible with a generator)
shuffled_indices: list[int] = torch.randperm(len(graphs), generator=torch.Generator().manual_seed(42)).tolist()
train_graphs: list[AtomicGraph] = [graphs[i] for i in shuffled_indices[:n_train]]
val_graphs: list[AtomicGraph] = [graphs[i] for i in shuffled_indices[n_train:]]

print(f"Training set:   {len(train_graphs)} structures")
print(f"Validation set: {len(val_graphs)} structures")

# ── Create DataLoaders ──────────────────────────────────────────────
batch_size: int = 16

train_loader: PyGDataLoader = PyGDataLoader(
    train_graphs,
    batch_size=batch_size,
    shuffle=True,       # shuffle every epoch for better training
)
val_loader: PyGDataLoader = PyGDataLoader(
    val_graphs,
    batch_size=batch_size,
    shuffle=False,       # no need to shuffle validation data
)

print(f"\nBatch size: {batch_size}")
print(f"Training batches per epoch:   {len(train_loader)}")
print(f"Validation batches per epoch: {len(val_loader)}")

In [ ]:
# ── Inspect one batch ────────────────────────────────────────────────
# Let's peek at what a batch looks like — this is exactly what the model
# receives during training.

sample_batch: AtomicGraph = next(iter(train_loader))

print("A single batch contains:")
print(f"  pos shape:        {sample_batch.pos.shape}     → all atoms from {batch_size} structures, stacked")
print(f"  z shape:          {sample_batch.z.shape}        → atomic numbers for all atoms")
print(f"  edge_index shape: {sample_batch.edge_index.shape}  → edges across all graphs")
print(f"  batch shape:      {sample_batch.batch.shape}    → which atom belongs to which graph")
print(f"  energy shape:     {sample_batch.energy.shape}   → one energy per structure")
print(f"  forces shape:     {sample_batch.forces.shape}  → forces for all atoms")
print(f"  num_graphs:       {sample_batch.num_graphs}")

# The 'batch' tensor is the key to understanding batching:
print(f"\nbatch tensor: {sample_batch.batch}")
print("  (each number tells you which graph that atom belongs to)")

---

## 7 · Configuring the Training Pipeline

Now we have a model and data — let's wire up the training. We need three things:

1. **Loss function** — tells the model how wrong its predictions are.
   We use a weighted combination of energy loss and forces loss.
2. **Optimiser** — the algorithm that adjusts model weights to reduce the loss.
   We use AdamW, a modern variant of gradient descent.
3. **`GOALModule`** — a Lightning module that puts it all together and handles
   the training loop, logging, checkpointing, etc.

### Understanding the loss function

For an interatomic potential, we want to be accurate on **both** energy and
forces. But forces are typically harder to learn and there are 3× more force
components than energy values, so we weight them more heavily:

$$\mathcal{L}_{\text{total}} = w_E \cdot \text{MSE}(E_{\text{pred}}, E_{\text{true}}) + w_F \cdot \text{MSE}(\mathbf{F}_{\text{pred}}, \mathbf{F}_{\text{true}})$$

Typical weights: $w_E = 4$, $w_F = 100$ (forces get 25× more weight).

In [ ]:
# ── Step 1: Build the loss function ─────────────────────────────────
# We compose two losses: one for energy, one for forces.
# Each is wrapped in a WeightedLoss with a scalar weight.

energy_loss: WeightedLoss = WeightedLoss(
    loss=EnergyLoss(loss_fn="mse"),  # Mean Squared Error on per-atom energy
    weight=4.0,                       # energy weight
    label="energy",                   # name for logging
)

forces_loss: WeightedLoss = WeightedLoss(
    loss=ForcesLoss(loss_fn="mse"),  # MSE on force components
    weight=100.0,                     # forces weight (higher = more emphasis on forces)
    label="forces",
)

# Combine them into a CompositeLoss
loss_fn: CompositeLoss = CompositeLoss([energy_loss, forces_loss])
print("Loss function:")
print(f"  Energy loss: MSE × {energy_loss.weight}")
print(f"  Forces loss: MSE × {forces_loss.weight}")

In [ ]:
# ── Step 2: Wrap in GOALModule ──────────────────────────────────────
# GOALModule is a Lightning module — it handles training_step,
# validation_step, optimiser setup, logging, and more.
#
# Normally it's created from a Hydra config, but here we build it
# manually to show what's happening under the hood.
#
# We need a minimal OmegaConf config for the parts GOALModule reads
# internally (optimiser settings, EMA, etc.)

from omegaconf import OmegaConf

# Minimal config — just enough for GOALModule to work
config = OmegaConf.create({
    "training": {
        "optimizer": {
            "lr": 0.001,               # learning rate
            "weight_decay": 0.0,        # L2 regularisation (0 = none)
            "amsgrad": True,            # AMSGrad variant of Adam
            "min_lr": 1e-7,             # minimum learning rate
            "scheduler_type": "reduce_on_plateau",
            "scheduler": {
                "patience": 10,         # reduce LR after 10 epochs with no improvement
                "factor": 0.5,          # halve the LR when reducing
            },
        },
        "ema": {"enabled": False},      # Exponential Moving Average (off for simplicity)
        "gradient_clip": 10.0,          # clip gradients to prevent explosions
    },
    "data": {"cutoff": cutoff},
    "model": {
        "backbone": {"name": "deepset", "cutoff": cutoff},
        "head": {"name": "energy_forces"},
    },
})

module: GOALModule = GOALModule(
    backbone=backbone,
    head=head,
    loss=loss_fn,
    config=config,
)

print("✓ GOALModule created")
print(f"  Backbone: {type(backbone).__name__}")
print(f"  Head:     {type(head).__name__}")
print(f"  Mode:     {'monolithic' if module.head is None else 'modular'}")

---

## 8 · Training the Model

Time to train! We'll use PyTorch Lightning's `Trainer`, which handles:
- The training loop (forward pass → loss → backward pass → optimiser step)
- Validation after each epoch
- Progress bars and logging
- Gradient clipping (prevents training from exploding)

We'll train for just **10 epochs** since this is a small demo dataset.
In real research, you'd train for 100–500 epochs with early stopping.

**What to watch for:**
- ✅ **Loss should decrease** over epochs — the model is learning
- ⚠️ **Train loss drops but val loss doesn't** — overfitting (model memorises data)
- ❌ **Loss increases or becomes NaN** — learning rate too high or bad data

In [ ]:
import lightning as L

# ── Create and run the Lightning Trainer ────────────────────────────
trainer: L.Trainer = L.Trainer(
    max_epochs=10,                     # short training for demo
    accelerator="cpu",                 # use "gpu" if you have one!
    devices=1,
    enable_checkpointing=True,         # save checkpoints
    default_root_dir="demo_training",  # output directory
    log_every_n_steps=1,               # log every step (small dataset)
    enable_progress_bar=True,
    num_sanity_val_steps=1,            # quick sanity check before training
)

# This one line does everything: train loop, validation, logging, checkpointing
trainer.fit(model=module, train_dataloaders=train_loader, val_dataloaders=val_loader)
print("\n✓ Training complete!")

In [ ]:
# ── Plot training progress ──────────────────────────────────────────
# Let's visualise how the loss changed over training.
# Lightning logs metrics automatically; we can read the logged values.

import matplotlib.pyplot as plt

# Extract logged metrics from the trainer
metrics: list[dict] = trainer.logged_metrics  # last epoch's metrics

# For a simple plot, let's re-evaluate to get per-epoch losses
# (Lightning's callback_metrics gives us the latest values)
print("Final training metrics:")
for key, value in trainer.callback_metrics.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.item():.6f}")
    else:
        print(f"  {key}: {value}")

---

## 9 · Saving and Loading a Trained Model Checkpoint

After training, you want to save your model so you can use it later
without retraining. Lightning creates checkpoints automatically (in the
`demo_training/` folder), but you can also save and load manually.

A **checkpoint** contains:
- All model weights (backbone + head)
- Optimiser state (so you can resume training)
- Hyperparameters
- The epoch number and best validation loss

In [ ]:
# ── Save the model manually ──────────────────────────────────────────
checkpoint_path: Path = Path("demo_model.ckpt")

# Method 1: Save just the model state dict (lightweight)
torch.save(
    {
        "backbone_state_dict": backbone.state_dict(),
        "head_state_dict": head.state_dict(),
        "config": OmegaConf.to_container(config),
    },
    checkpoint_path,
)
print(f"✓ Saved checkpoint to {checkpoint_path}")
print(f"  File size: {checkpoint_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# ── Load the model back from disk ────────────────────────────────────
# Create fresh model instances (as if we just started a new script)
loaded_backbone: nn.Module = DeepSet(
    num_elements=120, embedding_dim=64, hidden_channels=64,
    num_filters=64, num_radial_basis=20, transform_depth=2, cutoff=5.0,
)
loaded_head: nn.Module = EnergyForcesHead(irreps_in="64x0e", hidden_dim=32)

# Load the saved weights
ckpt: dict = torch.load(checkpoint_path, weights_only=False)
loaded_backbone.load_state_dict(ckpt["backbone_state_dict"])
loaded_head.load_state_dict(ckpt["head_state_dict"])

print("✓ Loaded model from checkpoint")

# Verify: run both on the same input and check outputs match
loaded_backbone.eval()
loaded_head.eval()
backbone.eval()
head.eval()

with torch.no_grad():
    test_graph: AtomicGraph = graphs[0]
    test_graph.pos.requires_grad_(True)

    orig_feats: NodeFeatures = backbone(test_graph)
    orig_preds: dict = head(orig_feats, test_graph)

    loaded_feats: NodeFeatures = loaded_backbone(test_graph)
    loaded_preds: dict = loaded_head(loaded_feats, test_graph)

    energy_match: bool = torch.allclose(orig_preds["energy"], loaded_preds["energy"])
    print(f"  Energy match: {energy_match} ✓" if energy_match else f"  Energy match: {energy_match} ✗")

---

## 10 · Using the Trained Model as an ASE Calculator

This is where the magic happens. You've trained a model — now you can use
it as an **ASE Calculator**, which means it plugs directly into the entire
[ASE ecosystem](https://wiki.fysik.dtu.dk/ase/):

- Single-point energy, forces, stress calculations
- Geometry optimisation (find minimum energy structures)
- Molecular dynamics (simulate atomic motion over time)
- Phonon calculations (vibrational properties)
- Nudged elastic band (reaction pathways)

GOAL provides `GOALCalculator` which wraps your trained module into the
standard ASE `Calculator` interface. You give it atoms → it gives you
energy and forces.

There are two ways to create a calculator:

| Method | When to use |
|--------|------------|
| `GOALCalculator(module=..., cutoff=...)` | You have the module in memory (like now) |
| `GOALCalculator(checkpoint_path="...")` | Loading from a saved Lightning checkpoint |

In [ ]:
# ── Create a GOALCalculator from our trained module ─────────────────
# We pass the GOALModule (which contains backbone + head + loss)
# and the cutoff radius used during training.

calc: GOALCalculator = GOALCalculator(
    module=module,    # our trained GOALModule
    cutoff=cutoff,    # must match the cutoff used for training!
    device="cpu",     # use "cuda" if you have a GPU
)

# ── Attach it to an ASE Atoms object and compute ───────────────────
test_atoms: Atoms = molecule("H2O")  # a fresh water molecule
test_atoms.calc = calc                # attach our ML calculator

# Now we can call standard ASE methods — the model runs under the hood!
energy_pred: float = test_atoms.get_potential_energy()  # eV
forces_pred: np.ndarray = test_atoms.get_forces()       # eV/Å

print(f"Predicted energy:  {energy_pred:.4f} eV")
print(f"Predicted forces (eV/Å):")
for i, (symbol, force) in enumerate(zip(test_atoms.get_chemical_symbols(), forces_pred)):
    print(f"  {symbol}: [{force[0]:+.4f}, {force[1]:+.4f}, {force[2]:+.4f}]")

**What just happened:**

1. We called `test_atoms.get_potential_energy()` — a standard ASE method
2. Under the hood, `GOALCalculator.calculate()` was called
3. It converted the ASE `Atoms` → `AtomicGraph` (building the neighbour list)
4. Ran the graph through our trained backbone → head pipeline
5. Extracted `energy` and `forces` from the output dictionary
6. Returned them in the format ASE expects

The model is still untrained on real data (our "DFT" energies were random!),
so the predictions are meaningless. But the pipeline is complete — with real
training data, you'd get physically meaningful results.

---

## 11 · Running a Simple ASE Calculation with the Calculator

Let's put the calculator to work with two common tasks:
1. **Geometry optimisation** — find the minimum energy structure
2. **Molecular dynamics** — simulate atomic motion

> **Note:** Since our model was trained on random data, the physics won't be
> meaningful. But the *code* is exactly what you'd use with a properly trained
> model on real DFT data.

In [ ]:
# ── Geometry Optimisation ────────────────────────────────────────────
# BFGS is a standard optimiser that moves atoms downhill on the energy
# surface until forces are below a threshold (fmax).

from ase.optimize import BFGS

opt_atoms: Atoms = molecule("H2O")
opt_atoms.calc = calc

print("Starting geometry optimisation...")
print(f"  Initial energy: {opt_atoms.get_potential_energy():.4f} eV")
print(f"  Initial max force: {np.max(np.abs(opt_atoms.get_forces())):.4f} eV/Å\n")

optimiser: BFGS = BFGS(opt_atoms, logfile=None)  # logfile=None suppresses verbose output

# Run up to 20 steps; stop if max force < 0.05 eV/Å
converged: bool = optimiser.run(fmax=0.05, steps=20)

print(f"\n  Final energy: {opt_atoms.get_potential_energy():.4f} eV")
print(f"  Final max force: {np.max(np.abs(opt_atoms.get_forces())):.4f} eV/Å")
print(f"  Steps taken: {optimiser.nsteps}")
print(f"  Converged: {converged}")

In [ ]:
# ── Molecular Dynamics ──────────────────────────────────────────────
# A short MD simulation using the Velocity Verlet integrator.
# We give the atoms thermal energy at 300 K and let them move.

from ase.md.verlet import VelocityVerlet
from ase import units

md_atoms: Atoms = molecule("H2O")
md_atoms.calc = calc

# Give atoms random velocities corresponding to 300 K
MaxwellBoltzmannDistribution(md_atoms, temperature_K=300)

# Create the dynamics object
# Timestep of 0.5 fs (femtosecond) — typical for atomistic MD
dyn: VelocityVerlet = VelocityVerlet(md_atoms, timestep=0.5 * units.fs)

# Collect energy at each step
md_energies: list[float] = []

def collect_energy() -> None:
    """Callback to record energy at each MD step."""
    md_energies.append(md_atoms.get_potential_energy())

dyn.attach(collect_energy, interval=1)

print("Running 50-step MD simulation at 300 K...")
dyn.run(steps=50)
print(f"✓ MD complete — collected {len(md_energies)} energy values")

# ── Plot the energy trajectory ──────────────────────────────────────
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for notebooks
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(md_energies, linewidth=1.5, color="#2196F3")
ax.set_xlabel("MD step")
ax.set_ylabel("Potential energy (eV)")
ax.set_title("Energy during MD simulation")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

---

## What's Next?

Congratulations — you've built, trained, and deployed an ML interatomic
potential entirely from Python code! Here's where to go from here:

| Goal | What to do |
|------|-----------|
| **Use real data** | Replace the synthetic `.traj` with your own DFT data |
| **Better models** | Try `InvariantGNN` or `HyperSpec` (equivariant) backbones |
| **Config-driven training** | Use `goal-train model=deepset data=trajectory` from the CLI |
| **Multi-GPU training** | `goal-train trainer=ddp` for DDP, or `trainer=fsdp` for FSDP |
| **Foundation models** | Fine-tune MACE or FairChem: `goal-finetune model.backbone.name=mace-large` |
| **Feature extraction** | See [`feature_extraction_demo.ipynb`](feature_extraction_demo.ipynb) |
| **Mini Trainer** | See [`mini_trainer_demo.ipynb`](mini_trainer_demo.ipynb) for a lightweight loop |
| **Full documentation** | Read the [README](../README.md) for comprehensive docs |

### Quick recap of the two model paradigms

```python
# ── Modular (recommended) ──────────────────────────────────────────
backbone = DeepSet(cutoff=5.0, hidden_channels=128, ...)
head = EnergyForcesHead(irreps_in="128x0e", hidden_dim=64)
module = GOALModule(backbone=backbone, head=head, loss=loss, config=config)

# ── Monolithic (for custom self-contained architectures) ───────────
model = MyMonolithicModel(cutoff=5.0, ...)
module = GOALModule(backbone=model, head=None, loss=loss, config=config)
```

Happy modelling! ⚛️